# Lab 06: Complex Data Types & JSON

## Objectives
- Work with struct, array, and map types in Spark
- Access nested fields in struct columns
- Use explode() to flatten arrays into rows
- Parse and manipulate JSON data from GitHub Archive events

## Exam Domain
- **Developing DataFrame/DataSet API Applications — 30%**

## Estimates
- **Time:** ~30 minutes
- **Cost:** ~$1 (serverless)
- **Compute:** Serverless

![Complex Types & Schema](../assets/diagrams/lab-06-complex-types-schema.png)

In [ ]:
USE_CATALOG = "spark_lab_guide"
USE_SCHEMA = "default"

spark.sql(f"USE CATALOG {USE_CATALOG}")
spark.sql(f"USE SCHEMA {USE_SCHEMA}")

github_df = spark.table("github_events")

## A. Inspecting Nested Schema

GitHub Archive events have deeply nested JSON structure. Let's explore the schema to understand the complex types.

In [ ]:
# Print the full schema — note StructType, ArrayType, and nested fields
github_df.printSchema()

In [ ]:
# Count events by type to understand the data
github_df.groupBy("type").count().orderBy("count", ascending=False).show()

## B. Accessing Struct Fields

Struct columns are like nested objects. Access their fields using dot notation.

In [ ]:
from pyspark.sql.functions import col

# Access nested struct fields with dot notation
github_df.select(
    col("id"),
    col("type"),
    col("actor.login").alias("username"),
    col("actor.id").alias("user_id"),
    col("repo.name").alias("repo_name"),
).show(10, truncate=False)

In [ ]:
# You can also use getField()
github_df.select(
    col("actor").getField("login").alias("username"),
    col("repo").getField("name").alias("repo_name"),
).show(5)

## C. Working with Arrays — explode()

Some fields contain arrays (lists of values). `explode()` creates one row per array element, flattening the structure.

In [ ]:
from pyspark.sql.functions import explode, size, col

# Filter to PushEvents which have commits arrays in payload
push_events = github_df.filter(col("type") == "PushEvent")

# Check the payload schema for PushEvents
push_events.select("payload.commits").printSchema()

In [ ]:
# Explode the commits array — one row per commit
exploded = (
    push_events
    .select(
        col("actor.login").alias("username"),
        col("repo.name").alias("repo"),
        explode(col("payload.commits")).alias("commit"),
    )
)

# Access fields within the exploded struct
commits = exploded.select(
    "username",
    "repo",
    col("commit.sha").alias("sha"),
    col("commit.message").alias("message"),
    col("commit.author.name").alias("author"),
)
commits.show(10, truncate=50)

In [ ]:
# Count commits per user
commits.groupBy("username").count().orderBy("count", ascending=False).show(10)

> **Exam Tip:** `explode()` creates new rows — if an array has 5 elements, you get 5 rows. For arrays that might be empty or null, use `explode_outer()` to keep the row with null instead of dropping it. The exam tests this distinction.

## D. explode_outer() vs explode()

In [ ]:
from pyspark.sql.functions import explode_outer

# Compare explode (drops nulls/empty) vs explode_outer (keeps nulls)
# Filter to events that may not have commits (non-PushEvents)
mixed_events = github_df.limit(100)

# explode drops rows where payload.commits is null
exploded_strict = mixed_events.select(
    "type",
    explode(col("payload.commits")).alias("commit"),
)
print(f"explode() rows: {exploded_strict.count()}")

# explode_outer keeps those rows with null
exploded_outer = mixed_events.select(
    "type",
    explode_outer(col("payload.commits")).alias("commit"),
)
print(f"explode_outer() rows: {exploded_outer.count()}")

## E. Creating and Working with Maps

Maps are key-value pairs. You can create them from columns or from arrays of keys and values.

In [ ]:
from pyspark.sql.functions import create_map, lit, map_keys, map_values

# Create a map column from existing columns
df_with_map = (
    github_df
    .select(
        col("id"),
        col("type"),
        create_map(
            lit("user"), col("actor.login"),
            lit("repo"), col("repo.name"),
        ).alias("metadata")
    )
)
df_with_map.show(5, truncate=False)

In [ ]:
# Access map values by key
df_with_map.select(
    col("metadata")["user"].alias("user"),
    col("metadata")["repo"].alias("repo"),
    map_keys(col("metadata")).alias("keys"),
    map_values(col("metadata")).alias("values"),
).show(5, truncate=False)

## F. collect_list() and collect_set()

These aggregate functions collect values into arrays — the reverse of explode.

In [ ]:
from pyspark.sql.functions import collect_list, collect_set

# Collect all event types per user (with duplicates)
user_events_list = (
    github_df
    .groupBy(col("actor.login").alias("username"))
    .agg(collect_list("type").alias("event_types"))
    .limit(5)
)
user_events_list.show(truncate=False)

# Collect unique event types per user (no duplicates)
user_events_set = (
    github_df
    .groupBy(col("actor.login").alias("username"))
    .agg(collect_set("type").alias("unique_event_types"))
    .limit(5)
)
user_events_set.show(truncate=False)

In [ ]:
# size() returns the length of an array or map
from pyspark.sql.functions import size

(
    github_df
    .groupBy(col("actor.login").alias("username"))
    .agg(
        collect_set("type").alias("unique_types"),
    )
    .withColumn("num_event_types", size("unique_types"))
    .orderBy(col("num_event_types").desc())
    .show(10, truncate=False)
)

## Exam-Style Review

**Q1.** How do you access a nested field `city` inside a struct column `address`?
- A) `col("address").city`
- B) `col("address.city")`
- C) `col("address")["city"]`
- D) Both B and using `getField("city")`

**Q2.** What does `explode()` do when the array column is null?
- A) Returns a row with null values
- B) Drops the row entirely
- C) Throws an error
- D) Returns an empty string

**Q3.** What is the difference between `collect_list()` and `collect_set()`?
- A) `collect_list()` returns unique values; `collect_set()` keeps duplicates
- B) `collect_list()` keeps duplicates and order; `collect_set()` returns unique values
- C) They are identical
- D) `collect_list()` returns a map; `collect_set()` returns an array

**Q4.** How do you access a value in a map column `m` with key `"name"`?
- A) `col("m").name`
- B) `col("m")["name"]`
- C) `col("m").getField("name")`
- D) `map_get(col("m"), "name")`

### Answers
- **Q1: D** — Both `col("address.city")` and `col("address").getField("city")` work for struct access.
- **Q2: B** — `explode()` drops rows where the array is null or empty. Use `explode_outer()` to keep them.
- **Q3: B** — `collect_list()` keeps duplicates and preserves insertion order. `collect_set()` returns unique values.
- **Q4: B** — Use bracket notation `col("m")["name"]` to access map values by key.

## Key Takeaways
- Access struct fields with dot notation: `col("struct.field")`
- `explode()` flattens arrays into rows (drops nulls); `explode_outer()` preserves nulls
- Maps are key-value pairs; access values with `col("map")["key"]`
- `collect_list()` keeps duplicates; `collect_set()` returns unique values
- `size()` returns the length of arrays and maps

**Next:** [Lab 07 — Spark SQL, Views & Catalog](07-spark-sql-views-catalog.ipynb)

In [ ]:
print("Lab 06 cleanup complete. No temp views to drop.")